# f6_m00c_export_probs.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M00c — Export de probabilidades |

---

## 🎯 Qué hace

Calcula la probabilidad de abandono predicha por el **modelo ganador activo**
para cada alumno del conjunto de test y la añade como columna `prob_abandono`
a `meta_test_app.parquet` (el fichero que usa la app Streamlit).

El nombre del modelo **no se hardcodea**: se lee de `metricas_modelo.json`
(clave `modelo_pkl`). Si el modelo ganador cambia, basta con regenerar el JSON
con `f6_m00_preparacion` y volver a ejecutar este notebook — sin tocar código.

Debe ejecutarse **después** de `f6_m00b_preparacion_app`. Si se vuelve a
ejecutar `m00b`, regenera el parquet desde cero y hay que volver a pasar `m00c`
para recuperar la columna `prob_abandono`.

## 📋 Requisitos

- `data/06_evaluacion/meta_test_app.parquet` — generado en `f6_m00b_preparacion_app`
- `data/05_modelado/X_test_prep.parquet` — features del test ya preprocesadas (Fase 5)
- `data/06_evaluacion/metricas_modelo.json` — contiene `modelo_pkl` (generado en `f6_m00_preparacion`)
- `data/05_modelado/models/<modelo_ganador>.pkl` — modelo entrenado en Fase 5, nombre leído del JSON
- Entorno: `tfm_abandono` (pandas, numpy, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/06_evaluacion/meta_test_app.parquet` | Mismo fichero sobrescrito con la columna `prob_abandono` añadida (6.725 × 36 columnas) |
| `data/06_evaluacion/meta_test_app.parquet.bak` | Copia de seguridad automática del estado anterior, por si la regeneración falla |
| `data/06_evaluacion/metricas_modelo.json` | Se le añaden dos claves de trazabilidad: `probs_modelo_pkl` y `probs_fecha_calculo` (qué modelo calculó las probabilidades y cuándo) |

## 🔄 Flujo

```
data/06_evaluacion/meta_test_app.parquet  ┐
data/05_modelado/X_test_prep.parquet      ├─→ predict_proba → prob_abandono
data/05_modelado/models/<ganador>.pkl     │
data/06_evaluacion/metricas_modelo.json   ┘
    ↓ comprobación de estado del parquet
    ↓ carga del modelo ganador (nombre leído del JSON, dinámico)
    ↓ cálculo de prob_abandono + verificación de calibración
    ↓ verificación con casos canónicos
    ↓ backup automático del parquet anterior
    → meta_test_app.parquet (con prob_abandono)
    → meta_test_app.parquet.bak
    → metricas_modelo.json (con huella del modelo)
```

## ➡️ Siguiente

`f6_m01a_shap_global.ipynb` — cálculo de valores SHAP sobre el conjunto de test

In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y RUTAS
# ============================================================================
import sys
from pathlib import Path

# Detección robusta de ROOT subiendo niveles hasta encontrar src/
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd

from src.utils.formatters import formato_numero_es, formato_porcentaje_es

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

ROOT: c:\PRUEBAS\AU_UJI_v2_RUTA_B
Python: 3.11.14


In [2]:
# ============================================================================
# CELDA 2: COMPROBACIÓN DE ESTADO DEL PARQUET
# ============================================================================
# Si `prob_abandono` ya existe, avisa (el notebook sobreescribe igualmente,
# pero así el usuario sabe que está reprocesando).
RUTA_META_APP = ROOT / "data" / "06_evaluacion" / "meta_test_app.parquet"

assert RUTA_META_APP.exists(), (
    f"❌ No encontrado: {RUTA_META_APP}\n"
    "   Ejecuta primero f6_m00b_preparacion_app.ipynb"
)

df_app = pd.read_parquet(RUTA_META_APP)
print(f"meta_test_app: {df_app.shape}")
print(f"Columnas actuales: {len(df_app.columns)}")

if "prob_abandono" in df_app.columns:
    print("\n⚠️ La columna 'prob_abandono' YA existe — se recalculará y sobrescribirá.")
    print(f"   Valores actuales: min={df_app['prob_abandono'].min():.4f}, "
          f"max={df_app['prob_abandono'].max():.4f}, "
          f"media={df_app['prob_abandono'].mean():.4f}")
else:
    print("\n✅ La columna 'prob_abandono' NO existe — se añadirá.")

meta_test_app: (6725, 35)
Columnas actuales: 35

✅ La columna 'prob_abandono' NO existe — se añadirá.


In [3]:
# ============================================================================
# CELDA 3: CARGAR X_test_prep + MODELO GANADOR (dinámico desde el JSON)
# ============================================================================
# X_test_prep.parquet contiene los 6.725 alumnos de test YA PREPROCESADOS
# por el pipeline de Fase 5 (imputación, escalado, codificación aplicados).
# Se puede pasar directamente a predict_proba sin transformaciones extra.
# Es la misma lógica que usa p02_titulacion._cargar_datos_app().
#
# DINAMISMO: el nombre del modelo NO se hardcodea.
# Se lee de metricas_modelo.json (clave 'modelo_pkl'), que es la fuente de
# verdad del sistema dinámico de selección de modelo ganador.
# Mismo patrón que app/config_app.py::_obtener_ruta_modelo_dinamico().
import json

RUTA_X_PREP = ROOT / "data" / "05_modelado" / "X_test_prep.parquet"
RUTA_JSON   = ROOT / "data" / "06_evaluacion" / "metricas_modelo.json"

# 1. Leer el JSON para saber qué modelo está activo HOY
assert RUTA_JSON.exists(), (
    f"❌ No encontrado: {RUTA_JSON}\n"
    "   Ejecuta primero f6_m00_preparacion.ipynb (genera el JSON con el "
    "modelo ganador del sistema dinámico)."
)
with open(RUTA_JSON, encoding="utf-8") as fh:
    meta_json = json.load(fh)

nombre_pkl = meta_json.get("modelo_pkl")
assert nombre_pkl, (
    f"❌ El JSON {RUTA_JSON.name} no contiene la clave 'modelo_pkl'.\n"
    "   Re-ejecuta f6_m00_preparacion.ipynb para regenerarlo correctamente."
)

# 2. Construir la ruta al .pkl y verificar que existe
RUTA_MODELO = ROOT / "data" / "05_modelado" / "models" / nombre_pkl

print(f"📌 Modelo activo según JSON: {nombre_pkl}")
print(f"   Familia: {meta_json.get('modelo_familia', '—')}")
print(f"   Estrategia: {meta_json.get('modelo_estrategia', '—')}")
print()

for ruta, nombre in [(RUTA_X_PREP, "X_test_prep.parquet"),
                     (RUTA_MODELO, nombre_pkl)]:
    assert ruta.exists(), f"❌ No encontrado: {nombre} en {ruta}"

X_prep = pd.read_parquet(RUTA_X_PREP)
modelo = joblib.load(RUTA_MODELO)

print(f"✅ X_test_prep cargado: {X_prep.shape}")
print(f"   Features ({len(X_prep.columns)}): {list(X_prep.columns)}")
print(f"\n✅ Modelo cargado: {type(modelo).__name__}")
# Verificación de seguridad: X_test_prep ya viene preprocesado de Fase 5.
# Si el .pkl fuera un Pipeline con pasos de preprocesado (scaler, encoder...),
# predict_proba volvería a transformar datos ya transformados → probs incorrectas.
# El modelo ganador actual es un Pipeline de 1 solo paso ('model'), correcto.
if hasattr(modelo, "named_steps"):
    pasos = list(modelo.named_steps.keys())
    print(f"   Pipeline de {len(pasos)} paso(s): {pasos}")
    assert pasos == ["model"], (
        f"⚠️  El Pipeline tiene pasos de preprocesado: {pasos}\n"
        f"    X_test_prep YA está preprocesado — pasarlo por este Pipeline\n"
        f"    aplicaría el preprocesado dos veces y las probabilidades serían\n"
        f"    incorrectas. Revisar: usar el estimador final (named_steps['model'])\n"
        f"    o cargar X_test sin preprocesar."
    )
    print(f"   ✅ Pipeline de 1 paso — predict_proba sobre X_test_prep es correcto")
else:
    print(f"   Estimador directo (sin Pipeline) — predict_proba sobre X_test_prep es correcto")

# Verificación: el número de alumnos debe coincidir con meta_test_app
assert len(X_prep) == len(df_app), (
    f"❌ Mismatch: X_prep tiene {len(X_prep)} filas, "
    f"df_app tiene {len(df_app)}"
)

# Verificación: índices idénticos (requisito para el merge final)
assert list(X_prep.index) == list(df_app.index), (
    "❌ Los índices de X_prep y df_app NO coinciden — "
    "re-ejecuta f6_m00_preparacion.ipynb y f6_m00b_preparacion_app.ipynb"
)
print("✅ Índices coinciden entre X_prep y df_app")


📌 Modelo activo según JSON: LightGBM__none.pkl
   Familia: Gradient Boosting
   Estrategia: none

✅ X_test_prep cargado: (6725, 27)
   Features (27): ['cred_superados_anio_1er', 'cupo', 'pais_nombre', 'provincia', 'universidad_origen', 'edad_entrada', 'anios_gap', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'via_acceso', 'rama', 'n_anios_beca', 'anios_sin_beca', 'situacion_laboral', 'n_anios_trabajando', 'max_pagos', 'orden_preferencia', 'cred_repetidos', 'tasa_repeticion', 'n_anios_sin_notas', 'tasa_abandono_titulacion', 'sexo', 'indicador_interrupcion', 'nota_1er_anio_missing', 'nota_acceso_missing', 'nota_selectividad_missing']

✅ Modelo cargado: Pipeline
   Pipeline de 1 paso(s): ['model']
   ✅ Pipeline de 1 paso — predict_proba sobre X_test_prep es correcto
✅ Índices coinciden entre X_prep y df_app


In [4]:
# ============================================================================
# CELDA 4: CALCULAR prob_abandono
# ============================================================================
# Llamada directa a predict_proba con X_test_prep. Devuelve una matriz
# de N x 2 (prob clase 0, prob clase 1). Nos quedamos con la columna 1
# (probabilidad de abandono = clase positiva).
probs = modelo.predict_proba(X_prep)[:, 1]

print(f"✅ Probabilidades calculadas: {len(probs)} valores")
print(f"   Rango: [{probs.min():.4f}, {probs.max():.4f}]")
print(f"   Media: {probs.mean():.4f}   Mediana: {np.median(probs):.4f}")
print(f"   Std:   {probs.std():.4f}")

# Verificación de calibración: la media de probs debe estar cerca de la
# tasa real de abandono del test (modelo bien calibrado ≈ media real).
tasa_real = df_app["abandono"].mean()
print(f"\n📊 Tasa real de abandono en test: {tasa_real:.4f}")
print(f"   Media de probs predichas:       {probs.mean():.4f}")
print(f"   Diferencia:                     {abs(probs.mean() - tasa_real):.4f}")

# Distribución por niveles de riesgo (umbrales de la app)
n_bajo  = int((probs < 0.30).sum())
n_medio = int(((probs >= 0.30) & (probs < 0.60)).sum())
n_alto  = int((probs >= 0.60).sum())
total   = len(probs)
print(f"\n📊 Distribución por niveles de riesgo:")
print(f"   Bajo  (< 30%):        {formato_numero_es(n_bajo)}  "
      f"({formato_porcentaje_es(n_bajo / total * 100)})")
print(f"   Medio (30-60%):       {formato_numero_es(n_medio)}  "
      f"({formato_porcentaje_es(n_medio / total * 100)})")
print(f"   Alto  (≥ 60%):        {formato_numero_es(n_alto)}  "
      f"({formato_porcentaje_es(n_alto / total * 100)})")

✅ Probabilidades calculadas: 6725 valores
   Rango: [0.0003, 0.9970]
   Media: 0.2946   Mediana: 0.0769
   Std:   0.3608

📊 Tasa real de abandono en test: 0.2925
   Media de probs predichas:       0.2946
   Diferencia:                     0.0021

📊 Distribución por niveles de riesgo:
   Bajo  (< 30%):        4.476  (66,6%)
   Medio (30-60%):       616  (9,2%)
   Alto  (≥ 60%):        1.633  (24,3%)


In [5]:
# ============================================================================
# CELDA 5: VERIFICACIÓN CON CASOS CANÓNICOS
# ============================================================================
# Los mismos 5 casos que usa m00b, pero ahora comprobamos la prob predicha.
# Casos con abandono=1 deberían tener prob alta; los de abandono=0 prob baja.
CASOS_CANONICOS = {
    "C1 — Ing. Informática (FP, abandona)":          15872,
    "C2 — Medicina (nota 11.94, no abandona)":       11906,
    "C3 — Comunicación (nota alta, abandona)":        14957,
    "C4 — Ing. Informática (mujer, no abandona)":     32472,
    "C5 — Derecho (mayor 25, trabaja, abandona)":      7176,
}

# Serie con las probs indexadas como X_prep (y por tanto como df_app)
probs_serie = pd.Series(probs, index=X_prep.index, name="prob_abandono")

print("Caso                                            | Real | Prob predicha")
print("-" * 78)
n_coherentes = 0
n_omitidos = 0
for nombre, idx in CASOS_CANONICOS.items():
    # Red de seguridad: si el test ha cambiado, el índice podría no existir.
    if idx not in df_app.index:
        n_omitidos += 1
        print(f"\n⚠️  CASO CANÓNICO NO ENCONTRADO")
        print(f"    Caso:   {nombre}")
        print(f"    Índice: {idx} — no existe en el conjunto de test actual")
        print(f"    El test actual tiene {len(df_app)} filas "
              f"(rango de índices {df_app.index.min()} – {df_app.index.max()}).")
        print(f"    Causa probable: el conjunto de test ha cambiado desde que se")
        print(f"                    fijaron estos índices (p.ej. un refiltrado de")
        print(f"                    datos en Fase 1-5 o una nueva ejecución del split).")
        print(f"    Acción: actualizar el diccionario CASOS_CANONICOS de esta celda")
        print(f"            con índices que sí existan en df_app.index.")
        print(f"    → El caso se omite; los demás casos y el guardado del parquet continúan.")
        continue
    real = df_app.loc[idx, "abandono"]
    prob = probs_serie.loc[idx]
    icono = "🔴" if real == 1 else "🟢"
    coherente = (real == 1 and prob >= 0.5) or (real == 0 and prob < 0.5)
    check = "✅" if coherente else "⚠️"
    if coherente:
        n_coherentes += 1
    print(f"{icono} {nombre:<45}|  {int(real)}   | {prob:.4f} {check}")

print("-" * 78)
n_total = len(CASOS_CANONICOS)
if n_omitidos == 0:
    print(f"Casos coherentes: {n_coherentes}/{n_total}")
    if n_coherentes == n_total:
        print("✅ Todos los casos canónicos son coherentes con el abandono real")
    else:
        print("⚠️ Algunos casos son incoherentes — revisar si es esperable (ej: perfiles ambiguos)")
else:
    print(f"Casos coherentes: {n_coherentes} · omitidos: {n_omitidos} · de {n_total} definidos")
    print("⚠️ Hay casos omitidos — revisar el diccionario CASOS_CANONICOS (ver avisos arriba).")

Caso                                            | Real | Prob predicha
------------------------------------------------------------------------------
🔴 C1 — Ing. Informática (FP, abandona)         |  1   | 0.5403 ✅
🟢 C2 — Medicina (nota 11.94, no abandona)      |  0   | 0.0048 ✅
🔴 C3 — Comunicación (nota alta, abandona)      |  1   | 0.0457 ⚠️
🟢 C4 — Ing. Informática (mujer, no abandona)   |  0   | 0.4306 ✅
🔴 C5 — Derecho (mayor 25, trabaja, abandona)   |  1   | 0.6577 ✅
------------------------------------------------------------------------------
Casos coherentes: 4/5
⚠️ Algunos casos son incoherentes — revisar si es esperable (ej: perfiles ambiguos)


In [6]:
# ============================================================================
# CELDA 6: AÑADIR COLUMNA, HACER BACKUP Y GUARDAR
# ============================================================================
# 1. Backup automático del parquet anterior (mitigación de riesgo).
#    Si esta celda falla, tienes meta_test_app.parquet.bak con el estado previo.
# 2. Sobrescribimos meta_test_app.parquet con la nueva columna prob_abandono.
#    Así el cargador de datos de la app lo lee sin cambios en su código.
# 3. Registramos huella en metricas_modelo.json (qué modelo se usó, cuándo).
#    Esto permite a la app detectar desincronizaciones futuras.
import shutil
from datetime import datetime

# --- 1. Backup automático antes de sobrescribir ---
RUTA_BAK = RUTA_META_APP.with_suffix(".parquet.bak")
if RUTA_META_APP.exists():
    shutil.copy2(RUTA_META_APP, RUTA_BAK)
    print(f"💾 Backup creado: {RUTA_BAK.name}  ({RUTA_BAK.stat().st_size / 1024:.1f} KB)")

# --- 2. Añadir columna y guardar ---
# Se redondea a 4 decimales por coherencia con el resto del proyecto
# (métricas, JSON y prints usan 4 decimales). Precisión más que suficiente
# para ordenar por riesgo o aplicar umbrales en la app.
df_app["prob_abandono"] = probs_serie.round(4)

# Verificación final: sin nulos en prob_abandono y shape esperado
assert df_app["prob_abandono"].isnull().sum() == 0, "❌ NaN en prob_abandono"
assert df_app.shape[0] == len(probs), "❌ Longitudes inconsistentes"
assert "prob_abandono" in df_app.columns, "❌ Columna no añadida"

df_app.to_parquet(RUTA_META_APP, index=True)

print(f"\n✅ Guardado: {RUTA_META_APP}")
print(f"   Shape:   {df_app.shape}")
print(f"   Tamaño:  {RUTA_META_APP.stat().st_size / 1024:.1f} KB")

# --- 3. Huella del modelo en metricas_modelo.json ---
# Añadimos dos claves de trazabilidad sin tocar el resto del JSON:
#   probs_modelo_pkl       → qué .pkl se usó para calcular las probs
#   probs_fecha_calculo    → cuándo se calcularon
# Si la app detecta que probs_modelo_pkl != modelo_pkl, sabe que el parquet
# está desincronizado y puede mostrar un warning al usuario.
meta_json["probs_modelo_pkl"]    = nombre_pkl
meta_json["probs_fecha_calculo"] = datetime.now().isoformat(timespec="seconds")

with open(RUTA_JSON, "w", encoding="utf-8") as fh:
    json.dump(meta_json, fh, ensure_ascii=False, indent=2)

print(f"\n🪪 Huella registrada en {RUTA_JSON.name}:")
print(f"   probs_modelo_pkl:    {nombre_pkl}")
print(f"   probs_fecha_calculo: {meta_json['probs_fecha_calculo']}")

# --- Resumen final ---
print(f"\n🎯 Columna prob_abandono lista para usar en la app:")
print(f"   Rango:   [{df_app['prob_abandono'].min():.4f}, {df_app['prob_abandono'].max():.4f}]")
print(f"   Media:   {df_app['prob_abandono'].mean():.4f}")
print(f"\n💡 Próximo paso: la app cargará automáticamente las nuevas probabilidades")
print(f"   al reiniciar el caché de Streamlit.")


💾 Backup creado: meta_test_app.parquet.bak  (216.0 KB)

✅ Guardado: c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\06_evaluacion\meta_test_app.parquet
   Shape:   (6725, 36)
   Tamaño:  242.6 KB

🪪 Huella registrada en metricas_modelo.json:
   probs_modelo_pkl:    LightGBM__none.pkl
   probs_fecha_calculo: 2026-05-14T18:42:52

🎯 Columna prob_abandono lista para usar en la app:
   Rango:   [0.0003, 0.9970]
   Media:   0.2946

💡 Próximo paso: la app cargará automáticamente las nuevas probabilidades
   al reiniciar el caché de Streamlit.
